# Understanding Hired Rides in NYC

_[Project prompt](https://docs.google.com/document/d/1VERPjEZcC1XSs4-02aM-DbkNr_yaJVbFjLJxaYQswqA/edit#)_

_This scaffolding notebook may be used to help setup your final project. It's **totally optional** whether you make use of this or not._

_If you do use this notebook, everything provided is optional as well - you may remove or add prose and code as you wish._

_Anything in italics (prose) or comments (in code) is meant to provide you with guidance. **Remove the italic lines and provided comments** before submitting the project, if you choose to use this scaffolding. We don't need the guidance when grading._

_**All code below should be consider "pseudo-code" - not functional by itself, and only a suggestion at the approach.**_

## Project Setup

In [5]:
# all import statements needed for the project, for example:

import os
import re
import bs4
import matplotlib.pyplot as plt
import pandas as pd
import requests
import sqlalchemy as db
import pyarrow.parquet as pq
import geopandas as gpd

In [6]:
# any constants you might need; some have been added for you, and 
# some you need to fill in

TLC_URL = "https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page"

TAXI_ZONES_DIR = ""
TAXI_ZONES_SHAPEFILE = f"{TAXI_ZONES_DIR}/taxi_zones.shp"
WEATHER_CSV_DIR = ""

CRS = 4326  # coordinate reference system

# (lat, lon)
NEW_YORK_BOX_COORDS = ((40.560445, -74.242330), (40.908524, -73.717047))
LGA_BOX_COORDS = ((40.763589, -73.891745), (40.778865, -73.854838))
JFK_BOX_COORDS = ((40.639263, -73.795642), (40.651376, -73.766264))
EWR_BOX_COORDS = ((40.686794, -74.194028), (40.699680, -74.165205))

DATABASE_URL = "sqlite:///project.db"
DATABASE_SCHEMA_FILE = "schema.sql"
QUERY_DIRECTORY = "queries"

In [7]:
# Make sure the QUERY_DIRECTORY exists
try:
    os.mkdir(QUERY_DIRECTORY)
except Exception as e:
    if e.errno == 17:
        # the directory already exists
        pass
    else:
        raise

## Part 1: Data Preprocessing

### Download Parquet files for Yellow Taxi & Uber trip data 

In [121]:
def get_html() -> str:
    response = requests.get(TLC_URL)
    html = response.content
    return html

def find_taxi_parquet_links():
    parquet_links = list()
    
    html = get_html()
    soup = bs4.BeautifulSoup(html, "html.parser")
    links = soup.find_all("a")
    pattern = re.compile(r"Yellow Taxi Trip Records")
    for link in links:
        title = link.get('title')
        if title != None:
            match = pattern.search(title)
            if match:
                parquet_links.append(link.get('href'))
    return parquet_links

def download_taxi_parquet_files():
    taxi_files = find_taxi_parquet_links()
    for file_url in taxi_files:
        file_url = file_url.replace(' ', '')
        name = file_url.split('trip-data/')[1]
        # Check if the file already exists
        if os.path.exists(name):
            pass
        else:
            response = requests.get(file_url, stream=True)
            with open(name, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024): 
                    if chunk:
                        f.write(chunk)  

In [122]:
download_taxi_parquet_files()

### Load Taxi Zones

In [8]:
def load_taxi_zones(shapefile):
    data = gpd.read_file(shapefile)
    return data

In [26]:
loaded_taxi_zones = load_taxi_zones('taxi_zones.shp')
loaded_taxi_zones

,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,geometry
0,1,0.116357,0.000782,Newark Airport,1,EWR,"POLYGON ((933100.918 192536.086, 933091.011 19..."
1,2,0.433470,0.004866,Jamaica Bay,2,Queens,"MULTIPOLYGON (((1033269.244 172126.008, 103343..."
2,3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx,"POLYGON ((1026308.77 256767.698, 1026495.593 2..."
3,4,0.043567,0.000112,Alphabet City,4,Manhattan,"POLYGON ((992073.467 203714.076, 992068.667 20..."
4,5,0.092146,0.000498,Arden Heights,5,Staten Island,"POLYGON ((935843.31 144283.336, 936046.565 144..."
...,...,...,...,...,...,...,...
258,259,0.126750,0.000395,Woodlawn/Wakefield,259,Bronx,"POLYGON ((1025414.782 270986.139, 1025138.624 ..."
259,260,0.133514,0.000422,Woodside,260,Queens,"POLYGON ((1011466.966 216463.005, 1011545.889 ..."
260,261,0.027120,0.000034,World Trade Center,261,Manhattan,"POLYGON ((980555.204 196138.486, 980570.792 19..."
261,262,0.049064,0.000122,Yorkville East,262,Manhattan,"MULTIPOLYGON (((999804.795 224498.527, 999824...."


In [41]:
loaded_taxi_zones.crs  

<Projected CRS: EPSG:2263>
Name: NAD83 / New York Long Island (ftUS)
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - New York - counties of Bronx; Kings; Nassau; New York; Queens; Richmond; Suffolk.
- bounds: (-74.26, 40.47, -71.8, 41.3)
Coordinate Operation:
- name: SPCS83 New York Long Island zone (US survey foot)
- method: Lambert Conic Conformal (2SP)
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [66]:
def lookup_coords_for_taxi_zone_id(zone_loc_id, loaded_taxi_zones):
    row = loaded_taxi_zones[loaded_taxi_zones['LocationID']==zone_loc_id]
    if (len(row)!=0):
        row = row.to_crs(2263)
        centroid = row.geometry.centroid.iloc[0]
        return centroid.x, centroid.y
    else:
        return None, None

### Calculate Sample Size (NOT SURE!!!)

In [67]:
def calculate_sample_size(p=0.5, e=0.05, z=1.96):
    n0 = ((z**2)*p*(1-p))/(e**2)
    #n = n0/(1 + (n0 - 1)/population)
    return n0

### Common Functions

In [68]:
def get_all_urls_from_tlc_page(taxi_page):
    response = requests.get(taxi_page)
    html = response.content
    soup = bs4.BeautifulSoup(html, "html.parser")
    a_links = soup.find_all("a")
    return a_links

In [69]:
def filter_parquet_urls(all_urls):
    parquet_links = list()
    pattern = re.compile(r"Yellow Taxi Trip Records|High Volume For-Hire Vehicle Trip Records")
    for link in all_urls:
        title = link.get('title')
        if title != None:
            match = pattern.search(title)
            if match:
                parquet_links.append(link.get('href'))
    return parquet_links

### Process Taxi Data

In [109]:
def get_and_clean_taxi_month(url):
    #get data
    file_url = url.replace(' ', '')
    name = file_url.split('trip-data/')[1]
    
    #only extract a sample size of data
    sample = int(calculate_sample_size(p=0.5, e=0.05, z=1.96))
    data = pq.read_table(name).to_pandas().sample(n=sample, random_state=1)
    
    #only extract specific columns we want (there are three types of dataframes)
    if ('VendorID' in data.columns):
        # use shp to find lat and lon for pickup/dropoff location
        drop_index = list()
        for index, row in data.iterrows():
            lon, lat = lookup_coords_for_taxi_zone_id(row['PULocationID'],loaded_taxi_zones)
            lon2, lat2 = lookup_coords_for_taxi_zone_id(row['DOLocationID'],loaded_taxi_zones)
            if (lon != None and lat != None and lon2 != None and lat2 != None):
                data['pickup_longitude'] = lon
                data['pickup_latitude'] = lat
                data['dropoff_longitude'] = lon2
                data['dropoff_latitude'] = lat2
            else:
                drop_index.append(index)
        data = data.drop(drop_index)
        data = data[['VendorID','tpep_pickup_datetime','tpep_dropoff_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']]                                      
    if ('vendor_id' in data.columns):
        data = data[['vendor_id','pickup_datetime','dropoff_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']]
    if ('vendor_name' in data.columns):
        data = data[['vendor_name','Trip_Pickup_DateTime','Trip_Dropoff_DateTime', 'Start_Lon', 'Start_Lat', 'End_Lon', 'End_Lat']]

    #normalize column names
    new_column_names = ['VendorID', 'pickup_datetime', 'dropoff_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
    data.columns = new_column_names

    #removing invalid data points (NaN)
    data = data.dropna()

    #normalize appropriate column types
    data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'])
    data['dropoff_datetime'] = pd.to_datetime(data['dropoff_datetime'])

    #removing trips that start and/or end outside (40.560445, -74.242330) and (40.908524, -73.717047)
    data = data[(data['pickup_latitude'] >= 40.560445) & (data['pickup_latitude'] <= 40.908524) &
                (data['pickup_longitude'] >= -74.242330) & (data['pickup_longitude'] <= -73.717047) &
                (data['dropoff_latitude'] >= 40.560445) & (data['dropoff_latitude'] <= 40.908524) &
                (data['dropoff_longitude'] >= -74.242330) & (data['dropoff_longitude'] <= -73.717047)]

    drop_index2 = list()
    for index, row in data.iterrows():
        if ((row['pickup_longitude']==row['dropoff_longitude']) and (row['pickup_latitude']==row['dropoff_latitude'])):
            drop_index2.append(index)
    data = data.drop(drop_index2)

    return data

In [110]:
def get_and_clean_taxi_data(parquet_urls):
    all_taxi_dataframes = []
    
    for parquet_url in parquet_urls:
        if ('yellow' in parquet_url):
            dataframe = get_and_clean_taxi_month(parquet_url)
            all_taxi_dataframes.append(dataframe)
        
    # create one gigantic dataframe with data from every month needed
    taxi_data = pd.concat(all_taxi_dataframes)
    return taxi_data

In [111]:
def get_taxi_data():
    all_urls = get_all_urls_from_tlc_page(TLC_URL)
    all_parquet_urls = filter_parquet_urls(all_urls)
    taxi_data = get_and_clean_taxi_data(all_parquet_urls)
    return taxi_data

In [112]:
taxi_data = get_taxi_data()

In [113]:
taxi_data

,VendorID,pickup_datetime,dropoff_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
285244,DDS,2010-01-26 19:19:39,2010-01-26 19:32:48,-73.985478,40.759372,-73.870775,40.774018
12605118,VTS,2010-01-17 11:57:00,2010-01-17 12:01:00,-73.968957,40.762998,-73.976062,40.756525
3569929,VTS,2010-01-30 15:26:00,2010-01-30 15:31:00,-73.956252,40.767900,-73.956012,40.779740
7312271,VTS,2010-01-27 08:21:00,2010-01-27 08:47:00,-73.981918,40.767830,-73.984875,40.767895
11346410,CMT,2010-01-02 03:52:49,2010-01-02 04:04:12,-74.003889,40.737609,-73.979442,40.727601
...,...,...,...,...,...,...,...
3219400,CMT,2009-12-17 21:41:20,2009-12-17 21:50:16,-73.992590,40.749356,-73.981057,40.762964
11146423,CMT,2009-12-10 17:42:08,2009-12-10 17:50:19,-73.985335,40.738492,-73.995626,40.726820
12994642,CMT,2009-12-19 00:42:44,2009-12-19 00:48:42,-73.993868,40.751170,-73.999803,40.761489
10742778,CMT,2009-12-17 07:42:37,2009-12-17 07:50:15,-73.964153,40.756483,-73.978035,40.757589


In [114]:
taxi_data.head()

,VendorID,pickup_datetime,dropoff_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
285244,DDS,2010-01-26 19:19:39,2010-01-26 19:32:48,-73.985478,40.759372,-73.870775,40.774018
12605118,VTS,2010-01-17 11:57:00,2010-01-17 12:01:00,-73.968957,40.762998,-73.976062,40.756525
3569929,VTS,2010-01-30 15:26:00,2010-01-30 15:31:00,-73.956252,40.767900,-73.956012,40.779740
7312271,VTS,2010-01-27 08:21:00,2010-01-27 08:47:00,-73.981918,40.767830,-73.984875,40.767895
11346410,CMT,2010-01-02 03:52:49,2010-01-02 04:04:12,-74.003889,40.737609,-73.979442,40.727601


In [115]:
taxi_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8887 entries, 285244 to 170745
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   VendorID           8887 non-null   object        
 1   pickup_datetime    8887 non-null   datetime64[ns]
 2   dropoff_datetime   8887 non-null   datetime64[ns]
 3   pickup_longitude   8887 non-null   float64       
 4   pickup_latitude    8887 non-null   float64       
 5   dropoff_longitude  8887 non-null   float64       
 6   dropoff_latitude   8887 non-null   float64       
dtypes: datetime64[ns](2), float64(4), object(1)
memory usage: 555.4+ KB


In [116]:
taxi_data.describe()

,pickup_datetime,dropoff_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
count,8887,8887,8887.000000,8887.000000,8887.000000,8887.000000
mean,2009-12-30 13:53:29.615393536,2009-12-30 14:05:20.958140928,-73.975802,40.752361,-73.974202,40.752126
min,2009-01-01 01:27:51,2009-01-01 01:42:03,-74.225003,40.603653,-74.183440,40.578006
25%,2009-06-30 03:58:43.500000,2009-06-30 04:10:30,-73.991657,40.737640,-73.991209,40.736376
50%,2009-12-30 07:30:00,2009-12-30 07:34:00,-73.981620,40.754798,-73.980182,40.754780
75%,2010-06-29 16:54:00,2010-06-29 16:59:30,-73.968799,40.768575,-73.965810,40.768769
max,2010-12-31 23:24:05,2010-12-31 23:30:03,-73.738470,40.894207,-73.721205,40.897803
std,NaN,NaN,0.032401,0.025856,0.033940,0.029854


### Processing Uber Data

In [ ]:
def get_and_clean_uber_month(url):
    raise NotImplementedError()

In [ ]:
def get_and_clean_uber_data(parquet_urls):
    all_uber_dataframes = []
    
    for parquet_url in parquet_urls:
        # maybe: first try to see if you've downloaded this exact
        # file already and saved it before trying again
        dataframe = get_and_clean_uber_month(parquet_url)
        # maybe: if the file hasn't been saved, save it so you can
        # avoid re-downloading it if you re-run the function
        
        all_uber_dataframes.append(dataframe)
        
    # create one gigantic dataframe with data from every month needed
    uber_data = pd.contact(all_uber_dataframes)
    return uber_data

In [ ]:
def load_and_clean_uber_data():
    raise NotImplementedError()

In [ ]:
def get_uber_data():
    all_urls = get_all_urls_from_tlc_page(TLC_URL)
    all_parquet_urls = find_parquet_urls(all_urls)
    taxi_data = get_and_clean_uber_data(all_parquet_urls)
    return taxi_data

In [ ]:
uber_data = get_uber_data()

In [ ]:
uber_data.head()

In [ ]:
uber_data.info()

In [ ]:
uber_data.describe()

### Processing Weather Data

In [ ]:
def get_all_weather_csvs(directory):
    raise NotImplementedError()

In [ ]:
def clean_month_weather_data_hourly(csv_file):
    raise NotImplementedError()

In [ ]:
def clean_month_weather_data_daily(csv_file):
    raise NotImplementedError()

In [ ]:
def load_and_clean_weather_data():
    weather_csv_files = get_all_weather_csvs(WEATHER_CSV_DIR)
    
    hourly_dataframes = []
    daily_dataframes = []
        
    for csv_file in weather_csv_files:
        hourly_dataframe = clean_month_weather_data_hourly(csv_file)
        daily_dataframe = clean_month_weather_data_daily(csv_file)
        hourly_dataframes.append(hourly_dataframe)
        daily_dataframes.append(daily_dataframe)
        
    # create two dataframes with hourly & daily data from every month
    hourly_data = pd.concat(hourly_dataframes)
    daily_data = pd.concat(daily_dataframes)
    
    return hourly_data, daily_data

In [ ]:
hourly_weather_data, daily_weather_data = load_and_clean_weather_data()

In [ ]:
hourly_weather_data.head()

In [ ]:
hourly_weather_data.info()

In [ ]:
hourly_weather_data.describe()

In [ ]:
daily_weather_data.head()

In [ ]:
daily_weather_data.info()

In [ ]:
daily_weather_data.describe()

## Part 2: Storing Cleaned Data

In [ ]:
engine = db.create_engine(DATABASE_URL)

In [ ]:
# if using SQL (as opposed to SQLAlchemy), define the commands 
# to create your 4 tables/dataframes
HOURLY_WEATHER_SCHEMA = """
TODO
"""

DAILY_WEATHER_SCHEMA = """
TODO
"""

TAXI_TRIPS_SCHEMA = """
TODO
"""

UBER_TRIPS_SCHEMA = """
TODO
"""

In [ ]:
# create that required schema.sql file
with open(DATABASE_SCHEMA_FILE, "w") as f:
    f.write(HOURLY_WEATHER_SCHEMA)
    f.write(DAILY_WEATHER_SCHEMA)
    f.write(TAXI_TRIPS_SCHEMA)
    f.write(UBER_TRIPS_SCHEMA)

In [ ]:
# create the tables with the schema files
with engine.connect() as connection:
    pass

### Add Data to Database

In [ ]:
def write_dataframes_to_table(table_to_df_dict):
    raise NotImplemented()

In [ ]:
map_table_name_to_dataframe = {
    "taxi_trips": taxi_data,
    "uber_trips": uber_data,
    "hourly_weather": hourly_data,
    "daily_weather": daily_data,
}

In [ ]:
write_dataframes_to_table(map_table_name_to_dataframe)

## Part 3: Understanding the Data

In [ ]:
# Helper function to write the queries to file
def write_query_to_file(query, outfile):
    raise NotImplementedError()

### Query 1

In [ ]:
QUERY_1_FILENAME = ""

QUERY_1 = """
TODO
"""

In [ ]:
# execute query either via sqlalchemy
with engine.connect() as con:
    results = con.execute(db.text(QUERY_1)).fetchall()
results

# or via pandas
pd.read_sql(QUERY_1, con=engine)

In [ ]:
write_query_to_file(QUERY_1, QUERY_1_FILENAME)

## Part 4: Visualizing the Data

### Visualization 1

In [ ]:
# use a more descriptive name for your function
def plot_visual_1(dataframe):
    figure, axes = plt.subplots(figsize=(20, 10))
    
    values = "..."  # use the dataframe to pull out values needed to plot
    
    # you may want to use matplotlib to plot your visualizations;
    # there are also many other plot types (other 
    # than axes.plot) you can use
    axes.plot(values, "...")
    # there are other methods to use to label your axes, to style 
    # and set up axes labels, etc
    axes.set_title("Some Descriptive Title")
    
    plt.show()

In [ ]:
def get_data_for_visual_1():
    # Query SQL database for the data needed.
    # You can put the data queried into a pandas dataframe, if you wish
    raise NotImplementedError()

In [ ]:
some_dataframe = get_data_for_visual_1()
plot_visual_1(some_dataframe)